# Colab: chạy tiếp (resume) arm `raw_aug_filt_loc`

Tiếp tục train từ `last.pt`, rồi eval trên val / FF++ test / Celeb-DF. **Chưa chạy thử trên Colab.**

**Chuẩn bị trên Google Drive** (đường dẫn sửa ở ô tham số):
1. `compffd/checkpoints/clip/raw_aug_filt_loc/` chứa `last.pt`, `best.pt`, `train_log.csv`, `run_info.json` (đúng thư mục run trên máy train).
2. `compffd/duy_output.zip`: nén thư mục `data_root` của preprocessing (có `ffpp/` gồm manifest, `params.json`, ảnh và **`aug/train/masks/`**, và `celebdf/`). Dữ liệu phải **giống hệt** lúc train (eval kiểm sha256 manifest).
3. Mã nguồn: clone từ GitHub (cần secret `GITHUB_TOKEN`) hoặc `compffd/detector.zip` (thư mục `detector/`).

**Runtime:** GPU. Mặc định code dùng autocast **bfloat16** (cần L4/A100, compute capability >= 8). Trên **T4** (không có bf16 gốc) notebook tự chuyển sang **fp16 + GradScaler** (`FFD_AMP=fp16`, cần nhánh `feat/amp-fp16`). Số học khác nhẹ so với phần đã train bằng bf16 (không bit-giống). Mỗi epoch ghi `last.pt` lên Drive; ngắt kết nối giữa chừng chỉ mất phần epoch đang dở (chưa có checkpoint giữa epoch). Ô 6 có bước đo tốc độ để ước lượng thời gian mỗi epoch trước khi chạy.

## 1. Tham số

In [ ]:
DRIVE      = "/content/drive/MyDrive"
CKPT_ROOT  = f"{DRIVE}/compffd/checkpoints/clip"      # chứa thư mục run bên dưới (lưu thẳng lên Drive để không mất khi ngắt)
RUN        = "raw_aug_filt_loc"
DATA_ZIP   = f"{DRIVE}/compffd/duy_output.zip"        # None nếu đã giải nén sẵn ở DATA_ROOT
DATA_ROOT  = "/content/duy_output"                    # đĩa local Colab (nhanh hơn đọc ảnh trực tiếp từ Drive)
REPO_ZIP   = None                                     # ví dụ f"{DRIVE}/compffd/detector.zip"; None -> clone GitHub
REPO_URL   = "https://github.com/duongduyy004/ComFFD_backbone.git"   # SỬA nếu host/đường dẫn khác (remote local là SSH alias)
BRANCH     = "feat/amp-fp16"                           # nhánh có tùy chọn FFD_AMP=fp16 (T4); "master" nếu dùng L4/A100
AMP        = "auto"                                   # auto | bf16 | fp16 ("auto": fp16 nếu compute capability < 8, tức T4)
REPO_DIR   = "/content/detector"
NUM_WORKERS = 2          # Colab free: 2 vCPU; Pro có thể tăng. Không vào config_hash.
GRAD_CKPT   = False      # đặt True nếu hết VRAM (chậm hơn ~21%). Không vào config_hash.
DATASETS    = "val,ffpp_test,celebdf"

## 2. Môi trường, Drive, GPU

In [ ]:
import os, sys, json, subprocess, pathlib, shutil, zipfile, time

def sh(cmd, cwd=None, check=True):
    """Chạy lệnh shell, in output theo dòng. Trả mã thoát."""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"lệnh lỗi (mã {rc}): {cmd}")
    return rc

from google.colab import drive
drive.mount("/content/drive")

import torch
assert torch.cuda.is_available(), "Cần runtime GPU: Runtime > Change runtime type"
name = torch.cuda.get_device_name(0); mem = torch.cuda.get_device_properties(0).total_memory / 2**30
cc = torch.cuda.get_device_capability(0)
print(f"GPU: {name}, {mem:.1f} GiB, compute capability {cc[0]}.{cc[1]}, torch {torch.__version__}")
if AMP == "auto":
    AMP = "bf16" if cc[0] >= 8 else "fp16"          # T4 = 7.5: không có bf16 gốc
os.environ["FFD_AMP"] = AMP                          # các lệnh con (ffd.cli) kế thừa biến này
print("autocast:", AMP, "(fp16 dùng GradScaler)" if AMP == "fp16" else "")
if mem < 12:
    print("VRAM < 12 GiB: nên đặt GRAD_CKPT = True ở ô tham số.")

## 3. Mã nguồn và thư viện

In [ ]:
if not pathlib.Path(REPO_DIR, "pyproject.toml").exists():
    if REPO_ZIP:
        with zipfile.ZipFile(REPO_ZIP) as z:
            z.extractall("/content/_repo_tmp")
        src = next(p for p in pathlib.Path("/content/_repo_tmp").rglob("pyproject.toml")).parent
        shutil.move(str(src), REPO_DIR)
    else:
        from google.colab import userdata
        tok = userdata.get("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{tok}@")
        sh(f"git clone --depth 1 -b {BRANCH} {url} {REPO_DIR}")
        sh(f"git remote set-url origin {REPO_URL}", cwd=REPO_DIR)   # không để token trong .git/config
print(subprocess.run("git log --oneline -1", shell=True, cwd=REPO_DIR, capture_output=True, text=True).stdout or "(không phải git repo)")

# Giữ torch của Colab; ghim các thư viện ảnh hưởng mô hình theo requirements.txt của repo.
sh("pip install -q -e . --no-deps", cwd=REPO_DIR)
sh("pip install -q transformers==5.22.2 peft==0.21.2 accelerate pyyaml tqdm scipy scikit-learn pyarrow opencv-python-headless")
sh("python -c \"import transformers, peft, ffd; print('transformers', transformers.__version__, 'peft', peft.__version__)\"")

## 4. Dữ liệu (giải nén ra đĩa local) và kiểm tra

In [ ]:
root = pathlib.Path(DATA_ROOT)
if not (root / "ffpp" / "manifest_raw.parquet").exists():
    assert DATA_ZIP, "Không có dữ liệu ở DATA_ROOT và DATA_ZIP=None"
    t0 = time.time()
    sh(f"mkdir -p /content/_data_tmp && unzip -q -o '{DATA_ZIP}' -d /content/_data_tmp")
    cand = next(p for p in pathlib.Path("/content/_data_tmp").rglob("manifest_raw.parquet")).parent.parent
    shutil.move(str(cand), DATA_ROOT)
    print(f"giải nén xong sau {time.time()-t0:.0f}s")
for f in ["ffpp/manifest_raw.parquet", "ffpp/manifest_aug.parquet", "ffpp/params.json", "celebdf/manifest.parquet"]:
    assert (root / f).exists(), f"thiếu {f}"
print("layout OK:", sorted(p.name for p in root.iterdir()))

## 5. Kiểm tra checkpoint và dựng config (khớp `config_hash` của run cũ)

Config lấy **nguyên từ `run_info.json` của checkpoint** (chỉ đổi đường dẫn, `num_workers`, `grad_checkpointing`; ba khóa này nằm ngoài hash). Dữ liệu phải trùng sha256 và thống kê lọc (`tau_m`, số ảnh) phải khớp lúc train.

In [ ]:
import yaml
sys.path.insert(0, REPO_DIR)
run_dir = pathlib.Path(CKPT_ROOT) / RUN
info = json.loads((run_dir / "run_info.json").read_text(encoding="utf-8"))
print("arm:", info["config"]["arm"], "| config_hash:", info["config_hash"], "| model_commit:", info["model_commit"][:8])
print("arm_constants:", info.get("arm_constants"))

cfg_d = dict(info["config"])
cfg_d.update(data_root=DATA_ROOT, ckpt_root=CKPT_ROOT, run_name=RUN, num_workers=NUM_WORKERS, grad_checkpointing=GRAD_CKPT)
CONFIG = "/content/config_resume.yaml"
pathlib.Path(CONFIG).write_text(yaml.safe_dump(cfg_d, allow_unicode=True), encoding="utf-8")

from ffd.config import load_train_config, config_hash
cfg = load_train_config(CONFIG)
assert config_hash(cfg) == info["config_hash"], f"config_hash khác ({config_hash(cfg)} != {info['config_hash']}): kiểm tra phiên bản code"
print("config_hash khớp.")

# dữ liệu phải trùng lúc train
from ffd.data.ffpp import data_hashes
h = data_hashes(DATA_ROOT)
assert h == info["data_hashes"], f"dữ liệu khác lúc train:\n  hiện tại {h}\n  lúc train {info['data_hashes']}"
print("sha256 dữ liệu khớp.")

# dựng TrainList: kiểm tra mask tồn tại và thống kê lọc trùng lúc train (không đọc ảnh)
from ffd.data.trainset import TrainList
ds = TrainList(DATA_ROOT, cfg.arm, cfg.seed, cfg.p_jpeg, cfg.jpeg_q)
for k in ("tau_m", "n_rep_kept", "n_seam_after"):
    assert abs(ds.aug_stats[k] - info["aug_stats"][k]) < 1e-12, (k, ds.aug_stats[k], info["aug_stats"][k])
print("TrainList OK:", len(ds), "ảnh; aug_stats khớp run cũ.")

# trạng thái run
print("done.flag:", (run_dir / "done.flag").exists(), "| result.json:", (run_dir / "result.json").exists(), "| last.pt:", (run_dir / "last.pt").exists())
if (run_dir / "last.pt").exists():
    ck = torch.load(run_dir / "last.pt", map_location="cpu", weights_only=False)
    print(f"last.pt: epoch {ck['epoch']}, step {ck['step']}, bad_epochs {ck['bad_epochs']}/{cfg.patience}, best {ck['best']}")
    del ck
print(subprocess.run(f"tail -n 4 '{run_dir}/train_log.csv' | cut -d, -f1-6", shell=True, capture_output=True, text=True).stdout)

## 6a. Đo tốc độ (tùy chọn nhưng nên chạy trên T4)

`smoke` chạy vài chục bước train thật để đo giây/bước và VRAM đỉnh, rồi ước lượng thời gian một epoch. Resume chỉ ở biên epoch, nên nếu một epoch dài hơn thời gian phiên Colab còn lại thì phần đó bị mất. Đo trên arm `raw_aug` (không có nhánh vị trí), nên ước lượng hơi thấp so với `filt_loc`.

In [ ]:
RUN_SMOKE = True
if RUN_SMOKE and not (run_dir / "result.json").exists() and not (run_dir / "done.flag").exists():
    sh(f"python -m ffd.cli smoke --no-progress --config {CONFIG} --steps 30", cwd=REPO_DIR)
    sm = json.loads((pathlib.Path(CKPT_ROOT) / "smoke" / "smoke.json").read_text(encoding="utf-8"))
    import math
    steps_per_epoch = math.ceil(len(ds) / cfg.batch_size)
    est_h = sm["sec_per_step"] * steps_per_epoch / 3600
    print(f"VRAM đỉnh {sm['peak_vram_gb']:.1f} GiB | {sm['sec_per_step']:.2f} s/bước | {steps_per_epoch} bước/epoch -> ước lượng ~{est_h:.1f} giờ/epoch")
    if est_h > 2:
        print("CẢNH BÁO: epoch dài; ngắt phiên giữa epoch sẽ mất tới ~%.1f giờ tiến độ. Cân nhắc GPU nhanh hơn." % est_h)

## 6. Train tiếp (resume)

Tự quyết định: có `result.json` thì bỏ qua; có `done.flag` thì chỉ eval; có `last.pt` thì `--resume`; chưa có gì thì train mới. Bộ đếm early stopping (`bad_epochs`), best, optimizer, scheduler và rng đều khôi phục từ `last.pt`. Nếu runtime ngắt, **chạy lại ô này**.

In [ ]:
if (run_dir / "result.json").exists():
    print("Đã có result.json: không train lại.")
elif (run_dir / "done.flag").exists():
    print("Train đã xong (done.flag): sang bước eval.")
else:
    if (run_dir / "last.pt").exists():
        flag = "--resume"
    elif any((run_dir / n).exists() for n in ("best.pt", "train_log.csv")):
        raise SystemExit("Có best.pt/train_log.csv nhưng không có last.pt: không resume được. Kiểm tra lại file đã chép.")
    else:
        flag = ""
    sh(f"python -m ffd.cli train --no-progress --config {CONFIG} {flag}", cwd=REPO_DIR)

## 7. Eval trên val / FF++ test / Celeb-DF

In [ ]:
if (run_dir / "result.json").exists():
    print("Đã có result.json, không eval lại.")
else:
    assert (run_dir / "done.flag").exists(), "Train chưa xong (không có done.flag); chạy lại ô 6."
    sh(f"python -m ffd.cli eval --no-progress --run '{run_dir}' --data-root {DATA_ROOT} --datasets {DATASETS}", cwd=REPO_DIR)
print((run_dir / "result.md").read_text(encoding="utf-8") if (run_dir / "result.md").exists() else "")

## 8. So sánh với các arm khác (tùy chọn) và xuất kết quả

So sánh theo cặp cần `<CKPT_ROOT>/raw/` và/hoặc `<CKPT_ROOT>/raw_aug_filt/` có `result.json` và `scores_*.parquet` (chép từ `results/`). Cặp thiếu run sẽ bị bỏ qua. CI của `compare` là bootstrap theo frame (lạc quan); chỉ số chính dùng bootstrap theo video.

In [ ]:
sh(f"python -m ffd.cli compare --ckpt-root '{CKPT_ROOT}'", cwd=REPO_DIR, check=False)

out = pathlib.Path(DRIVE) / "compffd" / "results" / RUN
out.mkdir(parents=True, exist_ok=True)
for n in ["result.json", "result.md", "run_info.json", "train_log.csv", "done.flag"]:
    if (run_dir / n).exists():
        shutil.copy2(run_dir / n, out / n)
for p in run_dir.glob("scores_*.parquet"):
    shutil.copy2(p, out / p.name)
print("Đã chép kết quả (không gồm best.pt/last.pt) tới:", out, sorted(x.name for x in out.iterdir()))